# Ejercicio 6: visualización de los vecinos de un aspártico

Los vecinos polares y no polares de dos aspárticos de la ribonucleasa (1LNI), con las mismas definiciones y funciones de `exercises/ex6.py`. Los números se corren con `venv/bin/python main.py 6iv` y `6v`; este notebook solo los dibuja.

Correr desde la raíz del TP con `venv/bin/jupyter lab <este archivo>`.

In [1]:
import nglview as nv
from structures import load

/Users/juliazack/Documents/fcen/bioinfo/bioinfo_tp5/venv/lib/python3.12/site-packages/nglview/__init__.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Parte 1: de la proteína al aminoácido

Antes de contar vecinos, cuatro dibujos para ubicar cada nivel: el archivo, la cadena, un aminoácido y sus puntos. Todos usan la ribonucleasa (1LNI).

In [2]:
from IPython.display import display
from contacts import POLAR, NONPOLAR, representative, residue_points
from exercises.ex6.ex6 import all_residues, bare_chain

rnase = load('1LNI')
chain = bare_chain('1LNI', 'A')   # la cadena A, sin agua ni hidrógenos: la misma que usa el ejercicio 6

# Para las selecciones de NGL: nombres de 3 letras de cada grupo de la consigna
POLAR3 = 'SER or THR or LYS or ARG or ASP or GLU or GLN or ASN'
NONPOLAR3 = 'ILE or LEU or VAL or TRP or TYR or PHE or MET'
CB_POINT = '(.CB or (GLY and .CA))'   # el punto CB de cada aminoácido (la glicina usa su CA)

### 1a. El archivo: dos copias de la proteína

El archivo 1LNI trae **dos copias** de la ribonucleasa, que quedaron juntas en el cristal: la **cadena A** (azul) y la **cadena B** (gris). Cada cadena es una proteína completa de 96 aminoácidos. El ejercicio 6 usa solo la A.

In [4]:
v = nv.show_biopython(rnase)
v.clear_representations()
v.add_representation('cartoon', selection=':A', color='steelblue')
v.add_representation('cartoon', selection=':B', color='lightgrey')
v

NGLWidget()

### 1b. La cadena: 96 aminoácidos enganchados

La cinta gris es la cadena A: los 96 aminoácidos enganchados uno detrás de otro, plegados en un ovillo. En rojo, **un solo aminoácido**: el 79, un aspártico (ASP). Es uno de los 96 eslabones de la cinta.

In [5]:
v = nv.show_biopython(rnase)
v.clear_representations()
v.add_representation('cartoon', selection=':A', color='lightgrey')
v.add_representation('ball+stick', selection=':A and 79', color='red')
v.center(selection=':A')
v

NGLWidget()

### 1c. Un aminoácido por dentro: eslabón, dije y sus tres puntos

Solo el aminoácido 79, de cerca, con el nombre de cada átomo. En **azul**, el eslabón (N, CA, C, O), igual en todos los aminoácidos; en **naranja**, el dije (cadena lateral), propio del aspártico: CB, CG, OD1, OD2. En gris, los eslabones de los vecinos 78 y 80, para ver que la cadena sigue para los dos lados.

Los tres puntos que usa el ejercicio son el **CA** y el **CB** (dos átomos, con su nombre) y el **CM**, la esfera violeta: el centro del dije. El CM no es un átomo, es un punto calculado (el promedio de las posiciones de los átomos del dije), por eso se dibuja aparte.

In [32]:
cm = [float(x) for x in representative(chain[79], 'CM')]
BACKBONE_SEL = '(.N or .CA or .C or .O)'

v = nv.show_biopython(rnase)
v.clear_representations()
v.add_representation('cartoon', selection=':A', color='lightgrey')
v.add_representation('licorice', selection=f':A and (78 or 80) and {BACKBONE_SEL}', color='lightgrey')
v.add_representation('ball+stick', selection=f':A and 79 and {BACKBONE_SEL}', color='blue')
v.add_representation('ball+stick', selection=f':A and 79 and not {BACKBONE_SEL}', color='orange')
v.add_representation('label', selection=':A and 79', labelType='atomname', color='black')
v.shape.add_sphere(cm, [0.6, 0.2, 0.8], 0.35, 'CM')
v.shape.add_text(cm, [0.6, 0.2, 0.8], 1.2, 'CM')
v.center(selection=':A and 79')
print('CM del aminoácido 79 en', [round(x, 2) for x in cm])
v

CM del aminoácido 79 en [56.43, 1.36, 5.49]


NGLWidget()

### 1d. Un punto por aminoácido

Ahora la cadena A reducida a **un punto por aminoácido** (su CB): 96 bolas, una por cada eslabón de la cinta. Es así como la "ve" el ejercicio 6 cuando cuenta vecinos. Colores según el grupo de la consigna: **azul** = polar, **naranja** = no polar, **gris** = ninguno de los dos (G, A, C, P, H).

In [33]:
v = nv.show_biopython(rnase)
v.clear_representations()
v.add_representation('cartoon', selection=':A', color='lightgrey', opacity=0.3)
v.add_representation('spacefill', selection=f':A and ({POLAR3}) and {CB_POINT}', color='blue', radius=1.0)
v.add_representation('spacefill', selection=f':A and ({NONPOLAR3}) and {CB_POINT}', color='orange', radius=1.0)
v.add_representation('spacefill', selection=f':A and not ({POLAR3} or {NONPOLAR3}) and {CB_POINT}', color='grey', radius=1.0)
residues, xyz = residue_points(chain, 'CB')
letters = [r.get_resname() for r in residues]
from Bio.SeqUtils import seq1
groups = [('polar' if seq1(n) in POLAR else 'no polar' if seq1(n) in NONPOLAR else 'otro') for n in letters]
print(f'{len(residues)} puntos:', {g: groups.count(g) for g in ('polar', 'no polar', 'otro')})
v

96 puntos: {'polar': 44, 'no polar': 28, 'otro': 24}


NGLWidget()

## Parte 2: la esfera alrededor de un aminoácido

Ahora el conteo de la parte iv. Elegimos un aminoácido (el 79, en **verde**) y dibujamos una esfera transparente centrada en su CB. Los aminoácidos cuyo punto CB cae **adentro** de la esfera se dibujan como bolas grandes, con el color de su grupo (azul polar, naranja no polar, gris otro); los que quedan **afuera**, como puntitos grises. El conteo de la tabla del ejercicio es cuántas bolas azules y naranjas hay adentro.

Se repite con los cuatro radios del ejercicio (6, 8, 10 y 12 Å): con más radio entran más vecinos. Los números que se imprimen se comparan contra la tabla del ejercicio 6, y la celda se detiene si no coinciden.

In [34]:
table = all_residues().set_index(['pdb', 'resseq'])

def show_sphere(resseq, radius, kind='CB'):
    residues, xyz = residue_points(chain, kind)
    i = [r.id[1] for r in residues].index(resseq)
    dist = np.linalg.norm(xyz - xyz[i], axis=1)
    inside = [residues[k] for k in range(len(residues)) if k != i and dist[k] <= radius]
    group = lambda r: 'polar' if seq1(r.get_resname()) in POLAR else 'nopolar' if seq1(r.get_resname()) in NONPOLAR else 'otro'
    by_group = {g: [r for r in inside if group(r) == g] for g in ('polar', 'nopolar', 'otro')}
    print(f'radio {radius:g} Å: {len(by_group["polar"])} polares, {len(by_group["nopolar"])} no polares, '
          f'{len(by_group["otro"])} de otro grupo (no se cuentan)')
    # Control: tiene que dar lo mismo que la tabla del ejercicio 6
    row = table.loc[('1LNI', resseq)]
    assert len(by_group['polar']) == row[f'polar_{kind}_{radius:g}']
    assert len(by_group['nopolar']) == row[f'nopolar_{kind}_{radius:g}']

    v = nv.show_biopython(rnase)
    v.clear_representations()
    v.add_representation('cartoon', selection=':A', color='lightgrey', opacity=0.2)
    v.add_representation('spacefill', selection=f':A and {CB_POINT}', color='grey', radius=0.4)
    for g, color in (('polar', 'blue'), ('nopolar', 'orange'), ('otro', 'grey')):
        if by_group[g]:
            nums = ' or '.join(str(r.id[1]) for r in by_group[g])
            v.add_representation('spacefill', selection=f':A and ({nums}) and {CB_POINT}', color=color, radius=1.0)
    v.add_representation('spacefill', selection=f':A and {resseq} and {CB_POINT}', color='green', radius=1.2)
    v.shape.add_sphere([float(x) for x in xyz[i]], [0.2, 0.8, 0.2], float(radius), 'esfera')
    v.update_representation(component=1, opacity=0.15)   # la esfera es el componente 1 (el 0 es la proteína)
    v.center(selection=f':A and {resseq}')
    return v

import numpy as np
from Bio.SeqUtils import seq1

In [35]:
show_sphere(79, 6)

radio 6 Å: 2 polares, 3 no polares, 0 de otro grupo (no se cuentan)


NGLWidget()

In [36]:
show_sphere(79, 8)

radio 8 Å: 4 polares, 8 no polares, 1 de otro grupo (no se cuentan)


NGLWidget()

In [37]:
show_sphere(79, 10)

radio 10 Å: 7 polares, 11 no polares, 3 de otro grupo (no se cuentan)


NGLWidget()

In [38]:
show_sphere(79, 12)

radio 12 Å: 14 polares, 18 no polares, 5 de otro grupo (no se cuentan)


NGLWidget()

Lo mismo se hace **para cada uno de los 96 aminoácidos**, cada uno con su propia esfera. Por ejemplo, el 25, con radio 8 Å: casi no tiene vecinos adentro.

In [39]:
show_sphere(25, 8)

radio 8 Å: 1 polares, 1 no polares, 2 de otro grupo (no se cuentan)


NGLWidget()

## Qué rodea a un aspártico: la pregunta del ejercicio 6, vista en 3D

El ejercicio 6 compara el pKa de cada aspártico (ASP) de la tabla con cuántos vecinos polares y no polares tiene alrededor. Acá dibujamos esos vecinos para dos casos opuestos de la ribonucleasa (1LNI, cadena A):

- **ASP 79**: pKa 7,37, el más alejado del valor normal (~4) de toda la tabla.
- **ASP 25**: pKa 4,87, cerca del valor normal.

Los vecinos se calculan con las mismas definiciones y el mismo código del ejercicio 6 (distancia entre CB, radio de 8 Å, mismos grupos polar/no polar de la consigna), así que los números que se imprimen son los de su tabla. Colores: el aspártico en **verde**, vecinos polares en **azul**, vecinos no polares en **naranja**; los vecinos que no están en ninguno de los dos grupos de la consigna (G, A, C, P, H) no se dibujan. La esfera transparente verde es la de 8 Å centrada en el CB del aspártico: los vecinos dibujados son los que tienen su CB adentro.

In [40]:
import numpy as np
from Bio.SeqUtils import seq1
from contacts import POLAR, NONPOLAR, representative, residue_points
from exercises.ex6.ex6 import bare_chain

RADIUS = 8.0   # Å, el mismo radio y el mismo punto (CB) que la tabla del ejercicio 6

def neighbors(pdb_id, chain_id, resseq, radius=RADIUS):
    """Vecinos polares y no polares de un residuo, con la definición del ejercicio 6."""
    residues, xyz = residue_points(bare_chain(pdb_id, chain_id), 'CB')
    i = [r.id[1] for r in residues].index(resseq)
    dist = np.linalg.norm(xyz - xyz[i], axis=1)
    near = [(residues[k], dist[k]) for k in np.argsort(dist) if k != i and dist[k] <= radius]
    polar = [(r, d) for r, d in near if seq1(r.get_resname()) in POLAR]
    nonpolar = [(r, d) for r, d in near if seq1(r.get_resname()) in NONPOLAR]
    for name, group in (('polares', polar), ('no polares', nonpolar)):
        print(f'{name} ({len(group)}): ' + ', '.join(f'{r.get_resname()}{r.id[1]} a {d:.1f} Å' for r, d in group))
    return [r for r, d in polar], [r for r, d in nonpolar]

def show_neighbors(pdb_id, chain_id, resseq):
    polar, nonpolar = neighbors(pdb_id, chain_id, resseq)
    v = nv.show_biopython(load(pdb_id))
    v.clear_representations()
    v.add_representation('cartoon', selection=f':{chain_id}', color='lightgrey', opacity=0.5)
    v.add_representation('ball+stick', selection=f':{chain_id} and {resseq}', color='green')
    for group, color in ((polar, 'blue'), (nonpolar, 'orange')):
        if group:
            numbers = ' or '.join(str(r.id[1]) for r in group)
            v.add_representation('licorice', selection=f':{chain_id} and ({numbers})', color=color)
    # Esfera transparente del mismo radio, centrada en el CB del residuo: los vecinos dibujados son los que caen adentro
    center = [float(x) for x in representative(bare_chain(pdb_id, chain_id)[resseq], 'CB')]
    v.shape.add_sphere(center, [0.2, 0.8, 0.2], float(RADIUS), 'esfera')
    v.update_representation(component=1, opacity=0.15)   # la esfera es el componente 1 (el 0 es la proteína)
    v.center(selection=f':{chain_id} and {resseq}')
    return v

In [41]:
show_neighbors('1LNI', 'A', 79)   # pKa 7,37: muy corrido

polares (4): THR72 a 5.5 Å, GLU78 a 5.8 Å, GLN77 a 7.3 Å, GLN94 a 7.5 Å
no polares (8): ILE92 a 4.4 Å, ILE70 a 5.5 Å, TYR80 a 5.8 Å, LEU19 a 6.8 Å, ILE71 a 7.0 Å, LEU91 a 7.3 Å, TYR81 a 7.3 Å, ILE22 a 8.0 Å


NGLWidget()

In [42]:
show_neighbors('1LNI', 'A', 25)   # pKa 4,87: casi normal

polares (1): SER24 a 5.6 Å
no polares (1): ILE22 a 7.5 Å


NGLWidget()